# 无线传输实验数据分析
---

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import leastsq

plt.rcParams['font.sans-serif'] = ['STZhongsong']
plt.rcParams['axes.unicode_minus'] = False

def sprint(name:str, sign:str, x:float, unit:str, original:bool = False, decimals:int = 4):
    units = {-4 : "p", -3 : "n", -2 : "μ", -1 : "m", 0 : "", 1 : "k", 2 : "M"}
    level = 0
    try:
        if np.abs(x) < 1:
            pass
        if not original:
            while np.abs(x) < 1:
                x = x * 1000
                level -= 1
            while np.abs(x) > 1000:
                x = x / 1000
                level += 1
        print(name + ": " + sign + " = %a" % np.around(x, decimals) + " " + units[level] + unit)
    except:
        if not original:
            while np.abs(x[0]) < 1:
                x = x * 1000
                level -= 1
            while np.abs(x[0]) > 1000:
                x = x / 1000
                level += 1
        print(name + ": " + sign + " = %a" % list(np.around(x, decimals)) + " " + units[level] + unit)

In [ ]:
# 可调电容最大值
C_0 = 1.7 * (10 ** -3) # H
# 线圈自感系数
L_0 = 460 * (10 ** -12) # F
# 采样电阻
R_ref = 68 # Ω
# 信号电动势峰值
V_0 = 20 # V
# 信号源内阻
R_0 = 50 # Ω

## 实验A

### A1

In [ ]:
f_0 = 1 / (2 * np.pi * np.sqrt(L_0 * C_0))

sprint("理论谐振频率", "f_0", f_0, "Hz")

### A2

In [ ]:
f_1 = 174.87 * (10 ** 3) # Hz
V_A_1 = 3.2046 # V
V_ref_1 = 1.6856 # V

f_2 = 175.57 * (10 ** 3) # Hz
V_A_2 = 3.1850 # V
V_ref_2 = 1.6856 # Vs

R_LC_1 = R_ref * (V_A_1 - V_ref_1) / V_ref_1
R_LC_2 = R_ref * (V_A_2 - V_ref_2) / V_ref_2

sprint("总寄生电阻（1）", "R_LC_1", R_LC_1, "Ω")
sprint("总寄生电阻（2）", "R_LC_2", R_LC_2, "Ω")

### A3

In [ ]:
phi_1 = np.arctan(R_LC_1 / (R_LC_1 + R_ref)) / (np.pi / 180)
phi_2 = np.arctan(R_LC_2 / (R_LC_2 + R_ref)) / (np.pi / 180)

sprint("需调整相位差（1）", "phi_1", phi_1, "°")
sprint("需调整相位差（2）", "phi_2", phi_2, "°")

In [ ]:
f_n_1 = 172.17 * (10 ** 3) # Hz
f_p_1 = 177.62 * (10 ** 3) # Hz
f_n_2 = 172.86 * (10 ** 3) # Hz
f_p_2 = 178.34 * (10 ** 3) # Hz

### A4

In [ ]:
Q_1 = f_0 / (f_p_1 - f_n_1)
Q_2 = f_0 / (f_p_2 - f_n_2)

sprint("品质因子（1）", "Q_1", Q_1, "")
sprint("品质因子（2）", "Q_2", Q_2, "")

In [ ]:
L_1 = Q_1 * R_LC_1 / (2 * np.pi * f_0)
L_2 = Q_2 * R_LC_2 / (2 * np.pi * f_0)

sprint("电感的自感系数（1）", "L_1", L_1, "H")
sprint("电感的自感系数（2）", "L_2", L_2, "H")

In [ ]:
C_1 = 1 / (2 * np.pi * f_0 * Q_1 * R_LC_1)
C_2 = 1 / (2 * np.pi * f_0 * Q_2 * R_LC_2)

sprint("电容（1）", "C_1", C_1, "F")
sprint("电容（2）", "C_2", C_2, "F")

## 实验B

### B1

In [ ]:
def cal_k(f1, f2):
    return (f2 ** 2 - f1 ** 2) / (f2 ** 2 + f1 ** 2)

In [ ]:
ss = np.arange(2.0, 12.0, 1.0) * (10 ** -2) # m
f1s = np.array([156.85, 160.88, 163.55, 165.96, 168.20, 169.18, 170.41, 171.71, 172.82, 174.16]) * (10 ** 3) # Hz
f2s = np.array([201.96, 194.43, 190.05, 186.57, 183.99, 182.16, 180.33, 178.80, 177.65, 174.41]) * (10 ** 3) # Hz
ks = np.array([cal_k(f1, f2) for f1, f2 in zip(f1s, f2s)])

sprint("线圈间距", "s", ss, "m")
sprint("谐振频率", "f1", f1s, "Hz")
sprint("谐振频率", "f2", f2s, "Hz")
sprint("耦合系数", "k", ks, "")

In [ ]:
fig = plt.figure(dpi = 160.0)
ax = fig.add_axes([0.1, 0.1, 0.8, 0.8])

plt.scatter(ss, ks, c = 'r', s = 10)
plt.plot(ss, ks, color = 'b', linewidth = 1, label = '$s-k$关系曲线')

def residuals_A(p):
    w, b = p
    return ks - (w * ss + b)
r = leastsq(residuals_A, [1, 0])
w, b = r[0]

XX = np.arange(0.015, 0.105, 0.001)
YY = w * XX + b
rho = np.corrcoef(ss, ks)
plt.plot(XX, YY, color = 'g', linewidth = 1, label = '使用最小二乘法拟合的$s-k$关系曲线')

ax.set_xlabel("线圈间距$s(cm)$")
ax.set_ylabel("耦合系数$k$")
ax.grid(visible = 1, which = 'major')
plt.title('线圈间距$s$和耦合系数$k$的关系\n拟合结果：$k=-%.6fs+%.6f$\t相关系数：$R^2=%.9f$' % (abs(w), abs(b), rho[0][1] ** 2))
plt.legend(loc = 'best')
plt.show()

## 实验C

### C1

In [ ]:
P_0_max = ((V_0 / (2 * R_0)) ** 2) * R_0 / 8

sprint("信号发生器最大平均输出功率", "P_0_max", P_0_max, "W")

### C2

In [ ]:
f_0_avg = (f_1 + f_2) / 2
L_0_avg = (L_1 + L_2) / 2
R_LC_avg = (R_LC_1 + R_LC_2) / 2

sprint("平均谐振频率", "f_0", f_0_avg, "Hz")
sprint("平均自感系数", "L_0", L_0_avg, "H")
sprint("平均总寄生电阻", "R_LC", R_LC_avg, "Ω")

In [ ]:
def cal_R_D(k):
    return (2 * np.pi * f_0_avg * k * L_0_avg) ** 2 / (R_0 + R_LC_avg) + R_LC_avg

In [ ]:
ks = np.array([cal_k(f1, f2) for f1, f2 in zip(f1s, f2s)])

R_Ds = np.array([cal_R_D(k) for k in ks])

sprint("线圈间距", "s", ss, "m")
sprint("耦合系数", "k", ks, "")
sprint("负载电阻理论值", "R_D", R_Ds, "Ω")

In [ ]:
fig = plt.figure(dpi = 160.0)
ax = fig.add_axes([0.1, 0.1, 0.8, 0.8])

plt.scatter(ss, R_Ds, c = 'r', s = 10)
plt.plot(ss, R_Ds, color = 'b', linewidth = 1, label = '$s-R_D$关系曲线')

ax.set_xlabel("线圈间距$s(cm)$")
ax.set_ylabel("负载电阻理论值$R_D(Ω)$")

ax.grid(visible = 1, which = 'major')
plt.title('线圈间距$s$和负载电阻理论值$R_D$的关系')
plt.legend(loc = 'best')
plt.show()

### C3

In [ ]:
def cal_eta(V_max, R_D):
    return V_max ** 2 / (8 * R_D * P_0_max)

In [ ]:
R_D_reals = np.array([2160, 1260, 820, 530, 340, 250, 170, 120, 90, 60]) # Ω
V_maxs = np.array([43.071, 32.781, 26.362, 20.825, 16.415, 13.573, 10.633, 8.477, 6.860, 5.047]) # V
f_reals = np.array([172.63, 173.35, 174.47, 175.81, 174.66, 175.76, 176.73, 176.73, 176.74, 176.74]) * (10 ** 3) # Hz
etas = np.array([cal_eta(V_max, R_D) for V_max, R_D in zip(V_maxs, R_D_reals)])

sprint("线圈间距", "s", ss, "m")
sprint("负载电阻理论值", "R_D", R_Ds, "Ω")
sprint("负载电阻实际值", "R_D_real", R_D_reals, "Ω")
sprint("电压最大峰值", "V_max", V_maxs, "V")
sprint("实际频率", "f_real", f_reals, "Hz")
sprint("传输效率", "η", etas, "", True)

In [ ]:
fig = plt.figure(dpi = 160.0)
ax = fig.add_axes([0.1, 0.1, 0.8, 0.8])

plt.scatter(ss, etas, c = 'r', s = 10)
plt.plot(ss, etas, color = 'b', linewidth = 1, label = '$s-η$关系曲线')

ax.set_xlabel("线圈间距$s(cm)$")
ax.set_ylabel("传输效率$\eta$")

ax.grid(visible = 1, which = 'major')
plt.title('线圈间距$s$和传输效率$\eta$的关系')
plt.legend(loc = 'best')
plt.show()

In [ ]:
fig = plt.figure(dpi = 160.0)
ax = fig.add_axes([0.1, 0.1, 0.8, 0.8])

plt.scatter(ks, etas, c = 'r', s = 10)
plt.plot(ks, etas, color = 'b', linewidth = 1, label = '$k-η$关系曲线')

ax.set_xlabel("线圈间的耦合系数$k$")
ax.set_ylabel("传输效率$\eta$")

ax.grid(visible = 1, which = 'major')
plt.title('线圈间的耦合系数$k$和传输效率$\eta$的关系')
plt.legend(loc = 'best')
plt.show()

In [ ]:
def cal_I_tp(V_max, R_D):
    return V_max / R_D

def cal_V_tp(V_max, I_tp, f):
    return np.sqrt(V_max ** 2 + (I_tp / (2 * np.pi * f * C_2)) ** 2)

In [ ]:
I_tps = np.array([cal_I_tp(V_max, R_D) for V_max, R_D in zip(V_maxs, R_D_reals)])
V_tps = np.array([cal_V_tp(V_max, I_tp, f) for V_max, I_tp, f in zip(V_maxs, I_tps, f_reals)])

sprint("线圈间距", "s", ss, "m")
sprint("电压最大峰值", "V_max", V_maxs, "V")
sprint("实际频率", "f_real", f_reals, "Hz")
sprint("传输的电流峰峰值", "I_tp", I_tps, "A")
sprint("传输的电压峰峰值", "V_tp", V_tps, "V")

In [ ]:
fig = plt.figure(dpi = 160.0)
ax1 = fig.add_axes([0.1, 0.1, 0.8, 0.8])
ax2 = ax1.twinx()

ax1.scatter(ss, I_tps, c = 'r', s = 10)
ax1.plot(ss, I_tps, color = 'b', linewidth = 1, label = '$s-I_{tp}$关系曲线')
ax2.scatter(ss, V_tps, c = 'c', s = 10)
ax2.plot(ss, V_tps, color = 'g', linewidth = 1, label = '$s-V_{tp}$关系曲线')

ax1.set_xlabel("线圈间距$s(cm)$")
ax1.set_ylabel("传输的电流峰峰值$I_{tp}(A)$")
ax2.set_ylabel("传输的电压峰峰值$V_{tp}(V)$")

plt.title('线圈间距$s$和传输的电流峰峰值$I_{tp}$、电压峰峰值$V_{tp}$的关系')
ax1.legend(loc = 2)
ax2.legend(loc = 4)
plt.show()

## 实验D

### D1

In [ ]:
f_best = R_0 / (2 * np.pi * np.sqrt(L_1 * L_2))

sprint("电能传输效率最大时的信号发生频率", "f_best", f_best, "Hz")

### D2

In [ ]:
Vs = np.array([2.2932 * (10 ** 3), 1.7346 * (10 ** 3), 1.3916 * (10 ** 3), 1.1466 * (10 ** 3), 960.4, 803.6, 666.4, 588.0, 509.2, 441.0]) / (10 ** 3) # V
_etas = np.array([cal_eta(V, R_0) for V in Vs])

sprint("线圈间距", "s", ss, "m")
sprint("负载电压最大峰值", "V", Vs, "V")
sprint("传输效率", "η", _etas, "", True, 6)

In [ ]:
fig = plt.figure(dpi = 160.0)
ax = fig.add_axes([0.1, 0.1, 0.8, 0.8])

plt.scatter(ss, _etas, c = 'r', s = 10)
plt.plot(ss, _etas, color = 'b', linewidth = 1, label = '$s-η$关系曲线')

ax.set_xlabel("线圈间距$s(cm)$")
ax.set_ylabel("传输效率$\eta$")

ax.grid(visible = 1, which = 'major')
plt.title('线圈间距$s$和传输效率$\eta$的关系')
plt.legend(loc = 'best')
plt.show()

In [ ]:
def cal_I_L(V):
    return V / R_0

def cal_V_L(V):
    return V

In [ ]:
I_Ls = np.array([cal_I_L(V) for V in Vs])
V_Ls = np.array([cal_V_L(V) for V in Vs])

sprint("线圈间距", "s", ss, "m")
sprint("负载电压最大峰值", "V", Vs, "V")
sprint("电流峰峰值", "I_L", I_Ls, "A")
sprint("电压峰峰值", "V_L", V_Ls, "V")

In [ ]:
sprint("线圈间距", "s", ss, "m")
sprint("LC耦合最大峰值", "η_D", etas, "", True)
sprint("互感耦合传输效率", "η_D", _etas, "", True)
sprint("效率比", "alpha", etas / _etas, "", True)

In [ ]:
fig = plt.figure(dpi = 160.0)
ax1 = fig.add_axes([0.1, 0.1, 0.8, 0.8])
ax2 = ax1.twinx()

ax1.scatter(ss, etas, c = 'r', s = 10)
ax1.plot(ss, etas, color = 'b', linewidth = 1, label = '$s-\eta_C$关系曲线')
ax1.scatter(ss, _etas, c = 'c', s = 10)
ax1.plot(ss, _etas, color = 'g', linewidth = 1, label = '$s-\eta_D$关系曲线')
ax2.plot(ss, V_tps, color = 'm', linewidth = 2, label = '$s-alpha$关系曲线')

ax1.set_xlabel("线圈间距$s(cm)$")
ax1.set_ylabel("传输效率$\eta$")
ax2.set_ylabel("效率比$alpha$")

plt.title('线圈间距$s$和传输效率比$alpha$的关系')
ax1.legend(loc = 1)
ax2.legend(loc = 2)
plt.show()